# MemAE-XGBoost IDS2 — Google Colab Runner

Pipeline end-to-end: **Clean → Split → Preprocess → MemAE → Features → XGBoost → Fusion → Reports**

**Yêu cầu trước khi chạy:**
- Source code đã được upload lên Google Drive tại `/MyDrive/IDS2/`
- Dataset CIC-IDS2017 (CSV files) nằm tại `/MyDrive/IDS2/data/cicids2017/`
- Runtime: **GPU (T4 hoặc A100)** — vào Runtime → Change runtime type → GPU

> ⚠️ Pipeline full (~6 families) mất khoảng **3–5 giờ** trên T4. Dùng `FAMILIES = 'botnet'` để test nhanh (~30 phút).

## 1. Mount Google Drive & Setup Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ── Đường dẫn tới project trên Drive ──
DRIVE_PROJECT = '/content/drive/MyDrive/IDS2'

# Symlink vào /content/IDS2 để dễ truy cập
PROJECT_DIR = '/content/IDS2'
if not os.path.exists(PROJECT_DIR):
    os.symlink(DRIVE_PROJECT, PROJECT_DIR)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Working directory:', os.getcwd())
!ls -la

## 2. Install Dependencies

In [ ]:
# Kiểm tra torch + CUDA trước
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU không khả dụng — kiểm tra Runtime → Change runtime type → GPU')

In [ ]:
# Cài các dependencies còn thiếu (torch đã có sẵn trên Colab)
!pip install \
    "pandas>=2.0" \
    "numpy>=1.24" \
    "pyarrow>=12.0" \
    "scikit-learn>=1.3,<2.0" \
    "xgboost>=2.0,<4.0" \
    "matplotlib>=3.7" \
    "seaborn>=0.12" \
    "pyyaml>=6.0" \
    "joblib>=1.3" \
    "tqdm>=4.65" \
    "scipy>=1.10" \
    -q
print('✓ Packages installed')

In [ ]:
# Verify imports
import pandas as pd, numpy as np, xgboost as xgb, sklearn
print(f'pandas={pd.__version__}  numpy={np.__version__}')
print(f'xgboost={xgb.__version__}  sklearn={sklearn.__version__}')

# Test project import
from src.utils.io import read_yaml
print('✓ Project imports OK')

## 3. Kiểm tra Dataset

In [ ]:
import glob

data_dir = os.path.join(PROJECT_DIR, 'data/cicids2017')
csv_files     = sorted(glob.glob(os.path.join(data_dir, '**/*.csv'),     recursive=True))
parquet_files = sorted(glob.glob(os.path.join(data_dir, '**/*.parquet'), recursive=True))

print(f'CSV files found    : {len(csv_files)}')
for f in csv_files:
    print(f'  {os.path.getsize(f)/1e6:7.1f} MB  {os.path.basename(f)}')

print(f'\nParquet files found: {len(parquet_files)}')
for f in parquet_files:
    print(f'  {os.path.getsize(f)/1e6:7.1f} MB  {os.path.basename(f)}')

if not csv_files and not parquet_files:
    print('⚠️  KHÔNG tìm thấy file! Hãy upload CSV/Parquet vào:')
    print(f'   {data_dir}/')

## 4. Cấu hình Pipeline

Chỉnh các tham số ở cell này trước khi chạy:

In [ ]:
# ════════════════════════════════════════════
# CẤU HÌNH — chỉnh ở đây
# ════════════════════════════════════════════

# Families để train/eval
# Options: 'all' | 'botnet' | 'dos' | 'ddos' | 'portscan' | 'web_attack' | 'brute_force'
# Dùng 'botnet' để test nhanh (~30 phút); 'all' để full run (~3-5 giờ)
FAMILIES = 'all'

# Force retrain dù artifact đã tồn tại
FORCE_RETRAIN = False     # True để train lại từ đầu

# Xóa data trung gian sau mỗi family để tiết kiệm Drive storage
CLEAN_DATA = True

# Configs
MEMAE_CONFIG   = 'configs/memae_targeted.yaml'
XGBOOST_CONFIG = 'configs/xgboost_zdr5.yaml'
WINDOW_CONFIG  = 'configs/window_features_zdr5.yaml'

# Experiment settings
EXPERIMENT_SUFFIX = 'host_disjoint_zdr5'
VARIANT_SUFFIX    = 'targetsel_zdr5'
FUSION_SUFFIX     = 'scorefusion'
BENCHMARK_MODE    = 'host_disjoint_window'
SPLIT_GROUP_MODE  = 'host'    # 'host' | 'exact_flow'
FPR_BUDGETS       = '0.001,0.005,0.01,0.02,0.05'
MAX_OBSERVED_FPR  = 0.05

# Preprocess transform backend: 'auto' dùng CUDA nếu Colab có GPU, fallback CPU nếu không có
PREPROCESS_DEVICE     = 'auto'   # 'auto' | 'cuda' | 'cpu'
PREPROCESS_BATCH_ROWS = 262144   # giảm xuống 131072 nếu bị CUDA OOM
PREPROCESS_TMP_DIR    = '/content/ids2_preprocess_tmp'  # tránh mmap trực tiếp lên Google Drive

# ════════════════════════════════════════════
print('Configuration:')
print(f'  families       = {FAMILIES}')
print(f'  force_retrain  = {FORCE_RETRAIN}')
print(f'  clean_data     = {CLEAN_DATA}')
print(f'  experiment     = {EXPERIMENT_SUFFIX}')
print(f'  variant        = {VARIANT_SUFFIX}')
print(f'  preprocess_dev = {PREPROCESS_DEVICE}')
print(f'  preprocess_tmp = {PREPROCESS_TMP_DIR}')

## 5. Chạy Full Pipeline

Output sẽ hiện realtime. GPU được sử dụng ở stage preprocess transform và MemAE khi CUDA khả dụng.

In [ ]:
import subprocess, time

# ── Build command ──────────────────────────────────────────
cmd = [
    sys.executable,
    '-u',   # QUAN TRỌNG: -u = unbuffered stdout → output hiện realtime
    'scripts/run_full_pipeline_all_families.py',
    '--families', FAMILIES,
    '--memae-config', MEMAE_CONFIG,
    '--xgboost-config', XGBOOST_CONFIG,
    '--window-config', WINDOW_CONFIG,
    '--experiment-suffix', EXPERIMENT_SUFFIX,
    '--variant-suffix', VARIANT_SUFFIX,
    '--fusion-suffix', FUSION_SUFFIX,
    '--benchmark-mode', BENCHMARK_MODE,
    '--split-group-mode', SPLIT_GROUP_MODE,
    '--fpr-budgets', FPR_BUDGETS,
    '--max-observed-test-fpr', str(MAX_OBSERVED_FPR),
    '--preprocess-device', PREPROCESS_DEVICE,
    '--preprocess-batch-rows', str(PREPROCESS_BATCH_ROWS),
    '--preprocess-tmp-dir', PREPROCESS_TMP_DIR,
]
if FORCE_RETRAIN:
    cmd.append('--force-retrain')
if CLEAN_DATA:
    cmd.append('--clean-data')

# ── Env: truyền CUDA vars + PYTHONPATH cho subprocess ─────
env = os.environ.copy()
env['PYTHONPATH']      = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONUNBUFFERED'] = '1'  # đảm bảo output không bị buffer

print('Command:', ' '.join(cmd))
print('─' * 60)

# ── Popen với live streaming output ───────────────────────
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # gộp stderr vào stdout
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - t0

print('─' * 60)
print(f'Exit code : {proc.returncode}')
print(f'Elapsed   : {elapsed/60:.1f} minutes')
if proc.returncode == 0:
    print('✅ Pipeline hoàn thành!')
else:
    print('❌ Pipeline thất bại — xem log ở trên để debug')

## 6. Xem Kết Quả

In [ ]:
import glob, json
from pathlib import Path
import pandas as pd

# Tìm report mới nhất
report_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'reports/run_*')))
if not report_dirs:
    print('Chưa có report nào!')
else:
    latest = report_dirs[-1]
    print(f'Latest run: {os.path.basename(latest)}\n')
    summaries = sorted(glob.glob(os.path.join(latest, '*summary.md')))
    for s in summaries:
        print(f'=== {os.path.basename(s)} ===')
        print(Path(s).read_text())
        print()

In [ ]:
# Parse kết quả thành DataFrame để dễ so sánh
all_rows = []
report_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'reports/run_*')))

for run_dir in report_dirs[-3:]:  # 3 runs gần nhất
    run_name = os.path.basename(run_dir)
    for summary_json in sorted(glob.glob(os.path.join(run_dir, '*summary.json'))):
        d = json.loads(Path(summary_json).read_text())
        for row in d.get('primary_results', []):
            all_rows.append({
                'run'           : run_name,
                'family'        : row.get('family'),
                'support'       : row.get('support'),
                'selected_model': row.get('selected_model'),
                'fpr_budget'    : row.get('fpr_budget'),
                'zdr'           : row.get('zdr'),
                'f1'            : row.get('f1'),
                'test_fpr'      : row.get('observed_test_fpr'),
                'status'        : row.get('status'),
            })

if all_rows:
    df = pd.DataFrame(all_rows)
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(df.sort_values(['run', 'family']))
else:
    print('Không tìm thấy summary JSON. Thử chạy pipeline trước.')

## 7. Chạy Từng Stage Riêng Lẻ (Resume)

Dùng khi Colab bị timeout giữa chừng — resume từ stage đã chạy xong.

In [ ]:
# ── Resume từ stage cụ thể ──
# STAGES: split | preprocess | memae | features | xgboost | fusion | reports

START_STAGE     = 'memae'    # bắt đầu từ stage này
STOP_STAGE      = 'reports'  # dừng sau stage này
RESUME_FAMILIES = 'all'

cmd_resume = [
    sys.executable, '-u',
    'scripts/run_full_pipeline_all_families.py',
    '--families', RESUME_FAMILIES,
    '--start-at', START_STAGE,
    '--stop-after', STOP_STAGE,
    '--memae-config', MEMAE_CONFIG,
    '--xgboost-config', XGBOOST_CONFIG,
    '--window-config', WINDOW_CONFIG,
    '--experiment-suffix', EXPERIMENT_SUFFIX,
    '--variant-suffix', VARIANT_SUFFIX,
    '--fusion-suffix', FUSION_SUFFIX,
    '--benchmark-mode', BENCHMARK_MODE,
    '--split-group-mode', SPLIT_GROUP_MODE,
    '--fpr-budgets', FPR_BUDGETS,
    '--max-observed-test-fpr', str(MAX_OBSERVED_FPR),
    '--preprocess-device', PREPROCESS_DEVICE,
    '--preprocess-batch-rows', str(PREPROCESS_BATCH_ROWS),
    '--preprocess-tmp-dir', PREPROCESS_TMP_DIR,
]

env = os.environ.copy()
env['PYTHONPATH']       = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONUNBUFFERED'] = '1'

print(f'Resuming: {START_STAGE} → {STOP_STAGE}')
print('─' * 60)
t0 = time.time()
proc = subprocess.Popen(
    cmd_resume, cwd=PROJECT_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit: {proc.returncode} | Elapsed: {(time.time()-t0)/60:.1f} min')

## 8. Kiểm tra Artifacts & Disk Usage

In [ ]:
# Disk usage
!df -h /content/drive/MyDrive 2>/dev/null || df -h .
print()

# Artifacts
from pathlib import Path
for subdir in ['artifacts/memae', 'artifacts/xgboost', 'artifacts/fusion',
               'data/processed', 'data/features', 'reports']:
    full = Path(PROJECT_DIR) / subdir
    if full.exists():
        files = list(full.rglob('*'))
        files = [f for f in files if f.is_file()]
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f'{subdir:35s}  {len(files):4d} files  {total_mb:8.1f} MB')
    else:
        print(f'{subdir:35s}  (not found)')

In [ ]:
# Xem training log của MemAE/XGBoost cho một family cụ thể
INSPECT_FAMILY  = 'botnet'    # ← đổi family muốn xem
INSPECT_VARIANT = VARIANT_SUFFIX

artifact_name = f'zero_day_{INSPECT_FAMILY}_{EXPERIMENT_SUFFIX}_{INSPECT_VARIANT}'

for kind in ['memae', 'xgboost']:
    log_path = Path(PROJECT_DIR) / 'artifacts' / kind / artifact_name / 'training_log.json'
    if log_path.exists():
        d = json.loads(log_path.read_text())
        print(f'\n── {kind.upper()} ({INSPECT_FAMILY}) ──')
        if kind == 'memae':
            for k in ['best_epoch', 'selection_metric', 'selection_fallback_reason',
                      'memory_diversity_weight', 'val_seen_attack_samples_used_for_sanity']:
                print(f'  {k}: {d.get(k)}')
            for split, s in d.get('reconstruction_stats', {}).items():
                if s:
                    print(f'  recon_{split}: mean={s["mean"]:.3f} p90={s["p90"]:.3f} p99={s["p99"]:.3f}')
        else:
            for k in ['best_iteration', 'best_score', 'feature_dims', 'model_feature_dims', 'scale_pos_weight']:
                print(f'  {k}: {d.get(k)}')
            fs = d.get('feature_selection', {})
            print(f'  feature_selection: {fs.get("selected_feature_count")}/{fs.get("original_feature_count")}')
    else:
        print(f'\n{kind}: not found at {log_path}')

## 9. Chạy Unit Tests

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'tests.test_pipeline_hardening', '-v'],
    cwd=PROJECT_DIR,
    env={**os.environ, 'PYTHONPATH': PROJECT_DIR, 'PYTHONUNBUFFERED': '1'},
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr[-3000:])  # chỉ in 3000 ký tự cuối stderr
print('Exit code:', result.returncode)